# NIH metrics

This notebook uses information extracted from [NIH Exporter](https://reporter.nih.gov/exporter) to identify NIH-NIBIB funded users of PhysioNet.

## Import packages

In [ ]:
import os
from pathlib import Path

import pandas as pd

from twentyfiveyears.nih import (combine_exporter_tables, get_physionet_users, get_investigators, get_authors, link_users)

## Setup

In [ ]:
# Set the base path
base_path = os.path.join("..", "data")

## Load map of Person IDs

All users are assigned a unique `person_id`.

In [ ]:
# Load the map of Person IDs
path = os.path.join(base_path, 'handcrafted', 'person_id_lookup.csv')
person_map = pd.read_csv(path)
person_map.head(3)

## Load PhysioNet dataset

Load a dataset containing the list of PhysioNet users

In [ ]:
# Load DataFrame of PhysioNet users
path = os.path.join(base_path, 'physionet', 'users.csv')
df_physionet_users = get_physionet_users(path, person_map, first_name_as_initial=True)
df_physionet_users.head(3)

## Load Principal Investigators of NIH projects

Load a list of Principal Investigators

In [ ]:
# Load the NIH project data
path = os.path.join(base_path, 'nih', 'exporter', 'projects')
df_projects = combine_exporter_tables(path, "RePORTER_PRJ_C_FY", start_year=1995)

### Limit the data to NIBIB projects

In [ ]:
df_projects = df_projects[df_projects['IC_NAME']=='National Institute of Biomedical Imaging and Bioengineering'.upper()]
df_projects.head(3)

In [ ]:
# Get the names of Principal Investigators
investigators = get_investigators(df_projects, first_name_as_initial=True)
investigators[0:3]

## Load authors of publications linked to NIH projects

Load a list of authors linked to NIH projects

In [ ]:
# Import the linking tables to connect publications / authors to a specific NIH institute
# NOTE: have to manually rename years 2016 - 2020 as RePORTER instead of REPORTER
path = os.path.join(base_path, 'nih', 'exporter', 'link_tables')
df_links = combine_exporter_tables(path, "RePORTER_PUBLNK_C_", start_year=1995)

In [ ]:
# Load the NIH publications data
path = os.path.join(base_path, 'nih', 'exporter', 'publications')
df_publications = combine_exporter_tables(path, "RePORTER_PUB_C_", start_year=1995)

### Limit the data to NIBIB publications 

In [ ]:
# First get the PROJECT_NUMBER from the df_links table
df_publications = pd.merge(df_publications, df_links, on='PMID')
# Next get merge with the projects DataFrame to get 'IC_NAME'
df_publications = pd.merge(df_publications, df_projects[['CORE_PROJECT_NUM', 'IC_NAME']].copy(), left_on='PROJECT_NUMBER', right_on='CORE_PROJECT_NUM')

In [ ]:
# Get a DataFrame only for NIBIB publications
# NOTE: this line isn't doing anything since we already filter the projects on this above and then merge on the associated CORE_PROJECT_NUM
df_publications = df_publications[df_publications['IC_NAME']=='National Institute of Biomedical Imaging and Bioengineering'.upper()]
df_publications.head(3)

In [ ]:
# At this point all remaining publications are NIBIB funded so we can drop duplicated publications
df_publications = df_publications.drop_duplicates(subset="PMID")

In [ ]:
# Get the names of authors
authors = get_authors(df_publications, first_name_as_initial=True)
authors[0:3]

## Match NIH listed people to PhysioNet users

Attempt to match people between the two sources

In [ ]:
# Match NIH Principal Investigators to PhysioNet users
# Set limit for testing
limit = None
df_physionet_users = link_users(df_physionet_users, investigators, match_group="investigators", limit=limit)

In [ ]:
# Match NIH authors to PhysioNet users
df_physionet_users = link_users(df_physionet_users, authors, match_group="authors", limit=limit)

In [ ]:
df_physionet_users.head(5)

Save the results

## Save the results

In [ ]:
# Save the results
save_path = os.path.join(base_path, 'physionet_users_nih_nibib_funded.csv')
path = Path(save_path)

# Convert to a path that works on the current OS
normalized_path = path.as_posix() if path.drive else Path(*path.parts).resolve()

# Output the merged DataFrame or save it to a file
df_physionet_users.to_csv(normalized_path, index=False)